**News Sentiment as a Trading Signal: Measuring Predictive Decay Across Holding Horizons**

Syed Sirajuddin · Master of Science in Applied Artificial Intelligence · Shiley Marcos School of Engineering, University of San Diego · AAI-590 Capstone

# Notebook 3 of 5 — Sentiment Model: Background and Fine-Tuning

This notebook covers the deep-learning core of the pipeline: a transformer-based financial sentiment classifier. It first situates the model in the existing literature (feeding the Literature Review section of the report), then documents the fine-tuning procedure on the Financial PhraseBank (the Model Training element of the code base), and finally defines the scoring interface used by the rest of the pipeline. Training metrics and test-set performance will be recorded after the training run.


## 1. Background: From Dictionaries to Domain-Specific Transformers

Early work on news sentiment in finance relied on word-count dictionaries. Loughran and McDonald (2011) demonstrated why general-purpose dictionaries fail in this domain: roughly three-quarters of the words the Harvard psychosocial dictionary marks as negative (e.g., *liability*, *tax*, *cost*) are not negative in financial usage, and they introduced finance-specific word lists that became the standard lexical baseline. Dictionary methods, however, cannot resolve context — "profits fell less than expected" counts a negative word yet conveys positive news.

Contextual language models address exactly this limitation. The transformer architecture (Vaswani et al., 2017) and its bidirectional pre-trained variant BERT (Devlin, Chang, Lee, & Toutanova, 2019) learn representations in which a word's meaning depends on its sentence, and Devlin et al. showed that a single pre-trained model could be fine-tuned to new classification tasks with modest labeled data. Araci (2019) applied this recipe to finance: **FinBERT**, a BERT model further pre-trained on financial text and fine-tuned on the Financial PhraseBank (Malo et al., 2014), improved state-of-the-art classification accuracy on that benchmark by roughly 15 percent over prior approaches, with the largest gains precisely on the context-dependent sentences where dictionaries fail. FinBERT and its descendants have since become the default sentiment engine in academic and practitioner studies of news-based trading.

This lineage motivates our choice directly: the project's signal quality is bounded by the sentiment model's ability to read financial language, and the published evidence indicates a domain-adapted transformer is the strongest available option at this scale.


In [1]:
# Environment setup: resolve the repository root so `src` imports work
# whether this notebook is run from notebooks/ or the project root.
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
RANDOM_SEED = 42

In [2]:
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

## 2. Model and Training Procedure

**Architecture.** The classifier is a BERT-base encoder: 12 transformer layers, 12 attention heads per layer, hidden size 768, approximately 110 million parameters, with a 3-way softmax classification head (*negative*, *neutral*, *positive*) attached to the [CLS] token representation. Two training modes are supported: fine-tuning `bert-base-uncased` from its generic pre-trained weights (which lets us report a full learning curve from scratch), or further fine-tuning the published `ProsusAI/finbert` checkpoint. The pre-trained checkpoint also serves as a fallback scorer so that a training-environment failure cannot block the downstream pipeline.

**Data.** The 75%-agreement subset of the Financial PhraseBank, split 80/10/10 (train/validation/test) with class stratification, exactly as prepared in Notebook 01.

**Optimization.** Cross-entropy loss; AdamW with learning rate 2×10⁻⁵, weight decay 0.01, and a 10% linear warm-up — the small learning rate is standard for fine-tuning, as larger rates catastrophically overwrite pre-trained weights (Devlin et al., 2019). Batch size 32, maximum sequence length 128 tokens (financial headlines and PhraseBank sentences are short), 3 epochs, fixed seed 42. Model selection uses **macro-averaged F1 on the validation set**, chosen over accuracy because the class distribution is imbalanced (neutral dominates) and because the trading signal depends on correctly identifying the *minority* positive and negative classes.

**Evaluation.** The held-out test split is scored exactly once, after model selection, and those figures are the ones quoted in the report.


In [3]:
# Fine-tune (GPU strongly recommended; ~10–15 min on a T4 in Google Colab).
# Equivalent CLI:  python -m src.sentiment.finetune --base bert-base-uncased
from src.sentiment.finetune import main as finetune

finetune(base_model="bert-base-uncased")

c:\Users\SyedM\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-03 21:35:12,145 TensorFlow version 2.21.0 available.
2026-08-03 21:35:16,258 HTTP Request: HEAD https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip "HTTP/1.1 302 Found"
2026-08-03 21:35:16,396 HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-03 21:35:16,561 HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-03 21:35:16,662 HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-03 21:35:16,760 HTTP Request: GET https://huggingface.

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.553757,0.387378,0.843478,0.762530
2,0.272621,0.181159,0.939130,0.931021
3,0.115064,0.160660,0.933333,0.926885


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.115064,0.214781,3,0.927746,0.901226


2026-08-03 21:35:56,576 Test metrics: {'test_loss': 0.21478091180324554, 'test_accuracy': 0.9277456647398844, 'test_macro_f1': 0.9012262748920855}
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.28s/it]


### 2.1 Training diagnostics

The training script saves the loss/validation-F1 curves to `reports/figures/finetune_curves.png`. The shape of these curves answers the report's over/underfitting question: training loss should fall smoothly while validation macro-F1 plateaus; a validation decline in later epochs would indicate overfitting, which the best-checkpoint selection guards against.

**Training results and test-set metrics — to be completed after the training run.** (Accuracy, macro-F1, and the per-class confusion matrix on the held-out test split will be recorded here.)


## 3. Scoring Interface for the Pipeline

Downstream stages need a single scalar per article rather than three class probabilities. We define

$$\text{score} = P(\text{positive}) - P(\text{negative}) \in [-1, 1],$$

which preserves both polarity and decisiveness: a confidently neutral article scores near zero, exactly as a trading signal should treat it. Notebook 04 additionally weights each article by its *confidence* (1 − P(neutral)) when aggregating to the daily level, so that ambiguous articles contribute less to the day's tone than decisive ones.


In [4]:
from src.sentiment.finbert import FinBertScorer
from src.config import PROCESSED_DIR

scorer = FinBertScorer()  # loads the fine-tuned checkpoint or ProsusAI/finbert

demo = ["Company X beats quarterly earnings expectations and raises guidance.",
        "Regulators open an investigation into Company X's accounting practices.",
        "Company X will hold its annual shareholder meeting on Tuesday."]
scorer.score_texts(demo).assign(text=demo)

2026-08-03 21:35:59,245 HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-03 21:35:59,270 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-08-03 21:35:59,370 HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-03 21:35:59,399 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-03 21:35:59,522 HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-03 21:35:59,633 HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-0

,p_positive,p_negative,p_neutral,score,text
0,0.944015,0.027630,0.028355,0.916385,Company X beats quarterly earnings expectation...
1,0.015456,0.734722,0.249822,-0.719266,Regulators open an investigation into Company ...
2,0.017429,0.045286,0.937285,-0.027857,Company X will hold its annual shareholder mee...


2026-08-03 21:36:00,429 HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
2026-08-03 21:36:00,533 HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-08-03 21:36:00,633 HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"


In [5]:
# Score the single canonical cleaned news table from Notebook 01 and persist
# for Notebooks 02, 04, and 06. The cleaned file already carries Alpha
# Vantage's own sentiment columns, which ride through scoring untouched.
news = pd.read_parquet(PROCESSED_DIR / "news_clean.parquet")

scored = scorer.score_news(news)
scored.to_parquet(PROCESSED_DIR / "news_scored.parquet", index=False)
print(f"Scored {len(scored):,} articles.")
print("Columns carried through:", [c for c in scored.columns if c.startswith('av_') or c in ('score','effective_date')])

Scored 95,758 articles.
Columns carried through: ['av_sentiment_score', 'av_relevance', 'effective_date', 'score']


---
### Acknowledgment of AI Tool Use

Portions of the code scaffolding and prose in this notebook were drafted with the assistance of Anthropic's Claude (Anthropic, 2026) and subsequently reviewed, tested, and revised by the author, who takes full responsibility for the final content, design decisions, and results. This acknowledgment is provided in accordance with University of San Diego academic integrity guidelines on the use of generative AI tools.

Anthropic. (2026). *Claude* [Large language model]. https://claude.ai

### References

Araci, D. (2019). *FinBERT: Financial sentiment analysis with pre-trained language models* (arXiv:1908.10063). arXiv. https://arxiv.org/abs/1908.10063

Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. In *Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies, Volume 1* (pp. 4171–4186).

Loughran, T., & McDonald, B. (2011). When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks. *The Journal of Finance, 66*(1), 35–65.

Malo, P., Sinha, A., Korhonen, P., Wallenius, J., & Takala, P. (2014). Good debt or bad debt: Detecting semantic orientations in economic texts. *Journal of the Association for Information Science and Technology, 65*(4), 782–796.

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. In *Advances in Neural Information Processing Systems, 30*.
